<a href="https://colab.research.google.com/github/ofir2207/Cloud-project/blob/main/ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
!pip install google-generativeai

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import firebase_admin
from firebase_admin import credentials, db

# Path to your service account key file
# Make sure you have uploaded this file to the 'Files' tab in Colab!
SERVICE_ACCOUNT_PATH = 'path/to/your/service-account-file.json'
DATABASE_URL = 'https://cloud-snake-group-default-rtdb.europe-west1.firebasedatabase.app/'

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred, {
        'databaseURL': DATABASE_URL
    })

In [ ]:
import requests
import re

# Public REST API URL for the inverted_index
DATABASE_URL = 'https://cloud-snake-group-default-rtdb.europe-west1.firebasedatabase.app/inverted_index.json'

def fetch_public_index():
    try:
        response = requests.get(DATABASE_URL)
        response.raise_for_status()
        data = response.json()

        firebase_patterns = []
        if data and isinstance(data, dict):
            for word in data.keys():
                # Create pattern to match whole word case-insensitively
                pattern = rf'\b{re.escape(word)}\b'
                response_text = f"Matched indexed keyword: '{word}'. How can I help you with this topic?"
                firebase_patterns.append((pattern, response_text))
            print(f"Successfully loaded {len(firebase_patterns)} keywords via REST API.")
        else:
            print("No data found or invalid format at the provided URL.")
        return firebase_patterns
    except Exception as e:
        print(f"Error fetching data: {e}")
        return []

new_patterns = fetch_public_index()

Successfully loaded 20 keywords via REST API.


In [ ]:
import os
import re
import nltk
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import userdata

nltk.download('punkt', quiet=True)

def build_system_prompt():
    return "You are a helpful chatbot specialized in tomato information and specific indexed keywords."

class GeminiChatbot:
    def __init__(self, patterns, model_name="gemini-1.5-flash"):
        api_key = userdata.get('GOOGLE_API_KEY')
        if not api_key:
            raise RuntimeError("Missing GOOGLE_API_KEY in Colab Secrets.")
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_name=model_name, system_instruction=build_system_prompt())
        self.chat = self.model.start_chat(history=[])
        self.patterns = patterns

    def get_pattern_response(self, text):
        # Check for keyword matches in the user input (case-insensitive)
        for pattern, response in self.patterns:
            if re.search(pattern, text, re.IGNORECASE):
                return response
        return None

    def reply(self, text: str) -> str:
        # Check Firebase patterns first
        pattern_res = self.get_pattern_response(text)
        if pattern_res:
            return f"[Firebase Match] {pattern_res}"

        # Fallback to Gemini AI
        try:
            r = self.chat.send_message(text)
            return (r.text or "").strip()
        except Exception as e:
            return f"Error calling Gemini: {e}"

# Initialize with the fetched patterns from the previous cell
current_patterns = new_patterns if 'new_patterns' in locals() else []
bot = GeminiChatbot(patterns=current_patterns)

# UI Setup
title = widgets.HTML("<h3>Firebase-Indexed Tomato Chatbot</h3>")
query_box = widgets.Text(placeholder="Ask about tomatoes or indexed keywords...", description="Query:", layout=widgets.Layout(width="80%"))
send_btn = widgets.Button(description="Send", button_style="primary")
output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="10px", height="300px", overflow="auto"))

history = []

def render():
    with output:
        clear_output(wait=True)
        for who, msg in history:
            print(f"{who}: {msg}\n")

def on_send(_):
    text = query_box.value.strip()
    if not text: return
    history.append(("You", text))
    query_box.value = ""
    render()
    ans = bot.reply(text)
    history.append(("Bot", ans))
    render()

send_btn.on_click(on_send)
display(title, widgets.HBox([query_box, send_btn]), output)
render()

HTML(value='<h3>Firebase-Indexed Tomato Chatbot</h3>')

Output(layout=Layout(border='1px solid #ddd', height='300px', overflow='auto', padding='10px'))